# ZeroED: Setup & Execution Guide (Kaggle / Google Colab)

This notebook sets up and runs **ZeroED** using **DeepSeek V4.1 Flash** (`deepseek-flash`) with **high reasoning effort** via the OpenAI-compatible API.
Defaults to the **Rayyan** dataset (or any dataset specified in step 2).

### Prerequisites
- **API Key**: Add your `DEEPSEEK_API_KEY` in Kaggle Secrets (*Add-ons* → *Secrets*) or Colab Secrets (*🔑 Secrets* tab).
- **Environment**: Standard Kaggle / Colab GPU or CPU instance (Kaggle recommended for fastText memory headroom).

### 1. Clone Repository & Install Dependencies

In [ ]:
import os, shutil

# Reset to root (/kaggle/working or /content) to prevent nested ZeroED/ZeroED
BASE_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(BASE_DIR)
TARGET_DIR = os.path.join(BASE_DIR, "ZeroED")

# If directory is corrupted or missing main.py, remove and re-clone cleanly
if os.path.isdir(TARGET_DIR) and not os.path.isfile(os.path.join(TARGET_DIR, "main.py")):
    shutil.rmtree(TARGET_DIR)

if not os.path.isdir(TARGET_DIR):
    !git clone --depth 1 https://github.com/TuanMinh21/ZeroED.git

%cd {TARGET_DIR}

# Maintain PyTorch compatibility (setuptools<82)
!pip install -q "setuptools<82" wheel

# Install fasttext (compiled, pre-built wheel, or conda fallback)
!pip install -q fasttext || pip install -q fasttext-wheel || conda install -y -q -c conda-forge fasttext

# Install remaining dependencies
!pip install -q -r requirements.txt

print("Repository and dependencies ready in:", os.getcwd())

### 2. Configure Dataset & DeepSeek API

In [ ]:
import os
from getpass import getpass

# Select dataset: 'rayyan' (default), 'hospital', 'beers', 'flights', etc.
DATASET = "rayyan"

# Retrieve DeepSeek API Key from Kaggle/Colab Secrets, or prompt user
api_key = ""
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
except Exception:
    try:
        from google.colab import userdata
        api_key = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        api_key = os.getenv("DEEPSEEK_API_KEY") or getpass("Enter DeepSeek API Key: ")

os.environ["LLM_API_KEY"] = api_key
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"
os.environ["LLM_MODEL"] = "deepseek-flash"          # DeepSeek V4.1 Flash
os.environ["LLM_REASONING_EFFORT"] = "high"         # High reasoning effort
os.environ["LLM_LOG_FILE"] = "llm_calls.jsonl"

print(f"Configured for dataset: {DATASET} | Model: {os.environ['LLM_MODEL']} | Reasoning Effort: {os.environ['LLM_REASONING_EFFORT']}")

### 3. Verify LLM Connection

In [ ]:
# Quick connectivity test with DeepSeek API
import utility

test_reply = utility.get_ans_from_llm("Respond with 'PONG' only.")
print("LLM Test Response:", test_reply.strip())

### 4. Prepare fastText Embeddings
Downloads `cc.en.300.bin` (once) and reduces dimensions to match the dataset column count.

In [ ]:
import glob

# Run fastText preparation script
!python scripts/prepare_fasttext.py $DATASET

# Export reduced fastText model path for main pipeline
model_files = glob.glob("ft_cache/cc.en.*.bin")
if model_files:
    os.environ["FASTTEXT_MODEL"] = os.path.abspath(model_files[0])
    print("Active FASTTEXT_MODEL:", os.environ["FASTTEXT_MODEL"])
else:
    print("Warning: Reduced fastText model not found in ft_cache/")

### 5. Update Run Configuration

In [ ]:
import yaml

# Update run_config.yaml with current dataset selection
with open("run_config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["datasets"] = [DATASET]
cfg["data"]["err_rate_list"] = ["01"]
cfg["model"]["api_use"] = True

with open("run_config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Updated run_config.yaml: target dataset = {DATASET}")

### 6. Execute ZeroED Pipeline

In [ ]:
# Run main ZeroED error detection pipeline
!python main.py --config run_config.yaml

### 7. Evaluation Results

In [ ]:
# Display evaluation metrics (Precision, Recall, F1)
result_files = glob.glob("result/pipeline/**/func_det_res.txt", recursive=True)
if result_files:
    latest_result = max(result_files, key=os.path.getmtime)
    print(f"Results summary from {latest_result}:\n")
    with open(latest_result) as f:
        print("".join(f.readlines()[:12]))
else:
    print("No results file found yet. Check logs for errors.")

### 8. Package & Download Outputs
Zips all experimental results, generated rules, and logs for local download.

In [ ]:
# Run packaging shell script
!bash scripts/package_output.sh $DATASET ..

# Trigger browser download in Colab or display link in Kaggle
zip_matches = glob.glob(f"../zeroed_output_{DATASET}_*.zip")
if zip_matches:
    target_zip = zip_matches[0]
    try:
        from google.colab import files
        files.download(target_zip)
    except Exception:
        from IPython.display import FileLink, display
        display(FileLink(target_zip))
else:
    print("No zip archive found.")